# WOURI — AXE-4 : Fine-tuning MMS-dyu

Fine-tune les **adapter layers** de `facebook/mms-1b-all` sur le Dioula ivoirien.

**Données** :
- 211 clips réels Common Voice dyu v24 (parole native)
- 37 WAVs TTS MMS dyu depuis corpus IVR Wourri (parole agricole)
- 79 717 phrases texte (Bayelemabaga + Jeli-ASR + Findora + IVR)

**Durée estimée** : ~20-30 min sur T4 (Colab gratuit)

**Résultat** : adapter `dyu` sauvegardé dans `/content/drive/MyDrive/wourri/models/mms-dioula-adapter/`

In [ ]:
# 1. Vérifier le GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'Non disponible — activer GPU dans Runtime > Modifier le type d\'exécution')

In [ ]:
# 2. Installer les dépendances
!pip install -q transformers==4.40.0 datasets evaluate jiwer accelerate soundfile librosa safetensors
print('Dépendances installées.')

In [ ]:
# 3. Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/wourri'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive monté. Répertoire Wourri : {DRIVE_ROOT}')

In [ ]:
# 4. Uploader les données depuis votre PC
# Uploader ces fichiers/dossiers dans /content/drive/MyDrive/wourri/ :
#   - data/dioula_dataset/           (généré par prepare_dioula_dataset.py)
#   - cv-corpus-24.0-2025-12-05-dyu/cv-corpus-24.0-2025-12-05/dyu/clips/ (295 MP3)
#   - wouri-api/data/tts_test_axe1/  (37 WAVs IVR)

# Vérification
import os
dataset_train = os.path.join(DRIVE_ROOT, 'data/dioula_dataset/train/metadata.jsonl')
cv_clips = os.path.join(DRIVE_ROOT, 'cv-corpus-24.0-2025-12-05-dyu/cv-corpus-24.0-2025-12-05/dyu/clips')

print('Dataset train:', 'OK' if os.path.exists(dataset_train) else 'MANQUANT — uploader data/dioula_dataset/')
print('CV clips:', 'OK (' + str(len(os.listdir(cv_clips))) + ' fichiers)' if os.path.exists(cv_clips) else 'MANQUANT (optionnel — sans audio CV)')

In [ ]:
# 5. Configuration
import os

DATASET_PATH = os.path.join(DRIVE_ROOT, 'data/dioula_dataset')
CV_CLIPS_DIR = os.path.join(DRIVE_ROOT, 'cv-corpus-24.0-2025-12-05-dyu/cv-corpus-24.0-2025-12-05/dyu/clips')
TTS_WAVS_DIR = os.path.join(DRIVE_ROOT, 'wouri-api/data/tts_test_axe1')
OUTPUT_DIR   = os.path.join(DRIVE_ROOT, 'models/mms-dioula-adapter')
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_MODEL  = 'facebook/mms-1b-all'
TARGET_LANG = 'dyu'
EPOCHS      = 4
BATCH_SIZE  = 8
LEARNING_RATE = 1e-3
MAX_AUDIO_SAMPLES = None  # None = toutes les paires audio disponibles

print('Config OK')
print(f'  Dataset  : {DATASET_PATH}')
print(f'  Output   : {OUTPUT_DIR}')
print(f'  Epochs   : {EPOCHS} | Batch : {BATCH_SIZE} | LR : {LEARNING_RATE}')

In [ ]:
# 6. Charger les paires audio+texte
import os, json, re
from pathlib import Path

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s'\u025b\u0254\u014b\u0272\u00e0\u00e1\u00e2\u00e8\u00e9\u00ea\u00ec\u00ed\u00ee\u00f2\u00f3\u00f4\u00f9\u00fa\u00fb]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

audio_records = []

# Source 1 : Common Voice dyu clips
cv_validated = os.path.join(DRIVE_ROOT, 'cv-corpus-24.0-2025-12-05-dyu/cv-corpus-24.0-2025-12-05/dyu/validated.tsv')
if os.path.exists(cv_validated):
    with open(cv_validated, encoding='utf-8') as f:
        lines = f.readlines()
    header = lines[0].strip().split('\t')
    path_idx = header.index('path')
    sent_idx = header.index('sentence')
    for line in lines[1:]:
        parts = line.strip().split('\t')
        if len(parts) <= max(path_idx, sent_idx): continue
        mp3 = os.path.join(CV_CLIPS_DIR, parts[path_idx])
        text = normalize_text(parts[sent_idx])
        if os.path.exists(mp3) and len(text.split()) >= 2:
            audio_records.append({'audio': mp3, 'text': text, 'source': 'cv_dyu', 'fmt': 'mp3'})
    print(f'CV dyu : {len(audio_records)} clips')
else:
    print('CV dyu : non disponible (uploader depuis Drive)')

# Source 2 : WAVs TTS IVR (AXE-1)
rapport_file = os.path.join(TTS_WAVS_DIR, 'rapport_axe1.json')
n_tts = 0
if os.path.exists(rapport_file):
    with open(rapport_file, encoding='utf-8') as f:
        rapport = json.load(f)
    for item in rapport.get('resultats', []):
        if item.get('status') != 'OK': continue
        wav = item.get('fichier_wav', '')
        text = normalize_text(item.get('phrase_bambara', ''))
        if os.path.exists(wav) and len(text.split()) >= 3:
            audio_records.append({'audio': wav, 'text': text, 'source': 'tts_ivr', 'fmt': 'wav'})
            n_tts += 1
    print(f'TTS IVR : {n_tts} WAVs')
else:
    print('TTS IVR : non disponible (optionnel)')

print(f'\nTotal paires audio+texte : {len(audio_records)}')

In [ ]:
# 7. Convertir MP3 -> WAV 16kHz (pour les clips CV)
import librosa, soundfile as sf, numpy as np
from pathlib import Path

WAV_CACHE = '/content/wav_cache'
os.makedirs(WAV_CACHE, exist_ok=True)

def load_audio_16k(path, fmt):
    """Charge n'importe quel audio et rééchantillonne à 16kHz."""
    audio, sr = librosa.load(path, sr=16000, mono=True)
    return audio  # np.float32 array @ 16kHz

# Pré-convertir les MP3 en WAV pour accélérer le training
print('Conversion MP3 -> WAV 16kHz...')
converted = []
for i, rec in enumerate(audio_records):
    if rec['fmt'] == 'mp3':
        wav_out = os.path.join(WAV_CACHE, f'cv_{i:04d}.wav')
        if not os.path.exists(wav_out):
            try:
                audio = load_audio_16k(rec['audio'], 'mp3')
                sf.write(wav_out, audio, 16000)
            except Exception as e:
                print(f'  Erreur {rec["audio"]}: {e}')
                continue
        rec['audio'] = wav_out
        rec['fmt'] = 'wav'
    converted.append(rec)
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{len(audio_records)}')

audio_records = converted
print(f'Conversion terminee : {len(audio_records)} paires audio prets')

In [ ]:
# 8. Construire le vocabulaire depuis toutes les données texte
import json
from pathlib import Path

all_sentences = [r['text'] for r in audio_records]

# Ajouter les phrases du dataset texte
train_jsonl = os.path.join(DATASET_PATH, 'train/metadata.jsonl')
if os.path.exists(train_jsonl):
    with open(train_jsonl, encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            all_sentences.append(row.get('sentence', ''))

# Vocabulaire
all_text = ' '.join(all_sentences)
vocab = sorted(set(all_text.replace(' ', '|')))
vocab_dict = {v: i for i, v in enumerate(vocab)}
if '|' not in vocab_dict: vocab_dict['|'] = len(vocab_dict)
if '[UNK]' not in vocab_dict: vocab_dict['[UNK]'] = len(vocab_dict)
if '[PAD]' not in vocab_dict: vocab_dict['[PAD]'] = len(vocab_dict)

vocab_path = os.path.join(OUTPUT_DIR, 'vocab.json')
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump({TARGET_LANG: vocab_dict}, f, ensure_ascii=False, indent=2)

print(f'Vocabulaire : {len(vocab_dict)} tokens -> {vocab_path}')
print(f'Caracteres speciaux dioula detectes : {[v for v in vocab_dict if v in "\u025b\u0254\u014b\u0272"]}')

In [ ]:
# 9. Charger le modèle MMS et initialiser les adapters
import torch
from transformers import (
    Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor, Wav2Vec2ForCTC
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# Tokenizer
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_path, unk_token='[UNK]', pad_token='[PAD]',
    word_delimiter_token='|', target_lang=TARGET_LANG,
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000,
    padding_value=0.0, do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(OUTPUT_DIR)

# Modèle
print(f'Chargement {BASE_MODEL}...')
model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    attention_dropout=0.0, hidden_dropout=0.0,
    feat_proj_dropout=0.0, layerdrop=0.0,
    ctc_loss_reduction='mean',
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)
model.init_adapter_layers()
model.freeze_base_model()
for param in model._get_adapters().values():
    param.requires_grad = True

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params totaux   : {total:,}')
print(f'Params adapters : {trainable:,} ({100*trainable/total:.1f}%)')
model = model.to(device)

In [ ]:
# 10. Préparer le HuggingFace Dataset (audio réel)
import soundfile as sf
from datasets import Dataset
from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: object
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{'input_values': f['input_values']} for f in features]
        label_features = [{'input_ids': f['labels']} for f in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors='pt')
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch['labels'] = labels
        return batch

def make_sample(rec):
    audio, _ = sf.read(rec['audio'])
    if audio.ndim > 1: audio = audio.mean(axis=1)  # stereo -> mono
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt')
    labels = processor(text=rec['text'], return_tensors='pt').input_ids
    return {
        'input_values': inputs.input_values[0].tolist(),
        'input_length': len(inputs.input_values[0]),
        'labels': labels[0].tolist(),
        'sentence': rec['text'],
    }

# Split train/test (90/10)
import random
random.seed(42)
random.shuffle(audio_records)
split_idx = int(len(audio_records) * 0.9)
train_recs = audio_records[:split_idx]
test_recs  = audio_records[split_idx:]

print(f'Preparation dataset : train={len(train_recs)} | test={len(test_recs)}')
print('Traitement en cours (peut prendre 2-3 min)...')

train_samples = [make_sample(r) for r in train_recs]
test_samples  = [make_sample(r) for r in test_recs]

train_dataset = Dataset.from_list(train_samples)
test_dataset  = Dataset.from_list(test_samples)

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)
print('Dataset pret.')

In [ ]:
# 11. Métrique WER
import numpy as np
import evaluate

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': round(wer, 4)}

print('Metrique WER chargee.')

In [ ]:
# 12. Fine-tuning !
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    group_by_length=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=EPOCHS,
    gradient_checkpointing=True,
    fp16=(device == 'cuda'),
    learning_rate=LEARNING_RATE,
    warmup_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=processor.feature_extractor,
)

print(f'Debut entraînement : {EPOCHS} epoques, lr={LEARNING_RATE}, batch={BATCH_SIZE}')
trainer.train()

In [ ]:
# 13. Sauvegarder l'adapter dyu
try:
    from safetensors.torch import save_file as safe_save
    from transformers.models.wav2vec2.modeling_wav2vec2 import WAV2VEC2_ADAPTER_SAFE_FILE
    adapter_filename = WAV2VEC2_ADAPTER_SAFE_FILE.format(TARGET_LANG)
    adapter_path = os.path.join(OUTPUT_DIR, adapter_filename)
    safe_save(model._get_adapters(), adapter_path, metadata={'format': 'pt'})
    print(f'Adapter sauvegarde : {adapter_path}')
except Exception as e:
    print(f'Safetensors echec ({e}), sauvegarde standard...')
    trainer.save_model()

print(f'\nModele complet sauvegarde dans : {OUTPUT_DIR}')
print('Contenu :')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f:<50} {size/1024:.1f} Ko')

In [ ]:
# 14. Test rapide : transcrire un clip CV dyu avec le nouvel adapter
import soundfile as sf
import torch

# Prendre le premier clip test
test_rec = test_recs[0] if test_recs else None
if test_rec:
    audio, _ = sf.read(test_rec['audio'])
    if audio.ndim > 1: audio = audio.mean(axis=1)

    inputs = processor(audio, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(pred_ids)[0]

    print(f'Fichier   : {os.path.basename(test_rec["audio"])}')
    print(f'Reference : {test_rec["text"]}')
    print(f'Prediction: {transcription}')
else:
    print('Aucun clip test disponible.')

## Prochaines étapes

1. **Télécharger** `models/mms-dioula-adapter/` depuis Google Drive vers votre PC
2. **Copier** dans `wourri/wouri-api/modeles_manuels/mms-dioula-adapter/`
3. **Modifier** `app/services/asr_soloni_nemo.py` pour charger l'adapter fine-tuné
4. **Tester** avec `tools/test_e2e_axe5.py`

**Résultat attendu** : WER réduit sur le Dioula ivoirien par rapport au modèle MMS de base.